### Data Extraction

In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *

spark.sql('USE mycatalog.myschema')

cur_user = 'ruturaj'

a_key = spark.sql(f'SELECT ACCESS_KEY FROM USER_KEYS WHERE USER_ID = "{cur_user}"').collect()[0][0]
s_key = spark.sql(f'SELECT SECRET_KEY FROM USER_KEYS WHERE USER_ID = "{cur_user}"').collect()[0][0]

click_stream_schema = StructType([

  StructField('user_id', StringType()),
  StructField('event_timestamp', StringType()),
  StructField('event_type', StringType()),
  StructField('product_id', StringType()),
  StructField('session_id', StringType()),
  StructField('order_status', StringType())

])

click_stream = spark.read.option("fs.s3a.access.key", a_key)\
                    .option("fs.s3a.secret.key", s_key)\
                    .option("fs.s3a.endpoint", "s3.amazonaws.com")\
                    .schema(click_stream_schema)\
                    .csv('s3://ruturaj-serverless/ecom_project/raw_data/clickStream/clickstream.csv', header = True)

orders_schema = StructType([

  StructField('order_id', StringType()),
  StructField('user_id', StringType()),
  StructField('product_id', StringType()),
  StructField('quantity', IntegerType()),
  StructField('order_price', DecimalType(12,2)),
  StructField('order_timestamp', StringType()),
  StructField('order_status', StringType()),

])

orders = spark.read.option("fs.s3a.access.key", a_key)\
                    .option("fs.s3a.secret.key", s_key)\
                    .option("fs.s3a.endpoint", "s3.amazonaws.com")\
                    .schema(orders_schema)\
                    .csv('s3://ruturaj-serverless/ecom_project/raw_data/orders/orders.csv', header = True)

products_schema = StructType([

  StructField('product_id', StringType()),
  StructField('product_name', StringType()),
  StructField('product_category', StringType()),
  StructField('product_price', DecimalType(12,2))

])

products = spark.read.option("fs.s3a.access.key", a_key)\
                    .option("fs.s3a.secret.key", s_key)\
                    .option("fs.s3a.endpoint", "s3.amazonaws.com")\
                    .schema(products_schema)\
                    .csv('s3://ruturaj-serverless/ecom_project/raw_data/products/products.csv', header = True)            


### Data Exploration

In [0]:
def explore_dataframe(df):

    if type(df).__name__ != 'DataFrame':
        raise TypeError(f"First parameter must be DataFrame, got {type(df).__name__}")

    columns = ['column_name', 'null_count', 'empty_count', 'data_type']
    rows = []

    for c, dtype in df.dtypes:

        row = []
        row.append(c)

        null_count = df.where(col(c).isNull()).count()
        row.append(null_count)

        if df.schema[c].dataType == StringType():

            empty_count = df.select(sum((col(c) == '').cast('int'))).collect()[0][0]

        elif (df.schema[c].dataType == IntegerType() or
              df.schema[c].dataType == FloatType() or
              df.schema[c].dataType == DoubleType() or
              isinstance(df.schema[c].dataType, DecimalType) or
              df.schema[c].dataType == LongType()):

            empty_count = df.select(sum((col(c) == 0).cast('int'))).collect()[0][0]

        else:

            empty_count = 0

        row.append(empty_count)

        row.append(dtype)

        row = tuple(row)
        rows.append(row)

    return spark.createDataFrame(rows, columns)

click_stream_explore = explore_dataframe(click_stream)
orders_explore = explore_dataframe(orders)
products_explore = explore_dataframe(products)

# orders_explore.display()
# products_explore.display()
# click_stream_explore.display()

### Data Cleaning

In [0]:
click_stream = click_stream.withColumn('event_timestamp', to_timestamp_ntz(col('event_timestamp')))\
                            .withColumn('click_date', to_date('event_timestamp'))\
                            .withColumn('click_time', date_format('event_timestamp', 'HH:mm:ss'))

orders = orders.withColumn('order_timestamp', to_timestamp_ntz(col('order_timestamp')))\
                .withColumn('order_date', to_date('order_timestamp'))\
                .withColumn('order_time', date_format('order_timestamp', 'HH:mm:ss'))

def clean_string_columns(df):

    if type(df).__name__ != 'DataFrame':
        raise TypeError(f"First parameter must be DataFrame, got {type(df).__name__}")

    df = df.distinct()

    for c, dtype in df.dtypes:

        new_col = c.strip()
        new_col = new_col.replace(' ', '_')
        new_col = new_col.replace('-', '_')

        if dtype == 'string':

            df = df.withColumnRenamed(c, new_col)
            df = df.withColumn(new_col, initcap(trim(col(new_col))))
            df.fillna({new_col: ''})

        elif dtype == 'int':

            df.fillna({new_col: 0})

    return df

click_stream = clean_string_columns(click_stream)
orders = clean_string_columns(orders)
products = clean_string_columns(products)